In [4]:
import pandas as pd

# read the CSV file
df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/data_optim_V_MES.csv')  #

# Filter conditions: year == 2023, compensatedRelativePressure_mafiltered is not NaN, and alpha != 0
filtered_df = df[(df['year'] == 2023) & 
                 (df['compensatedRelativePressure_mafiltered'].notna()) &
                (df['alpha'] != 0)] 

# Step 3: Create a new DataFrame (grouped by csemCur, ensuring non-empty parts for each csemCur)
new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))

# load heritage data
donnees_patrimoniales = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/Donnees patrimoniales - volume.csv')

# convert csemCur to numeric for both DataFrames
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce')
donnees_patrimoniales['csemCur'] = pd.to_numeric(donnees_patrimoniales['csemCur'], errors='coerce')

# merge Pression Nominale Relative column based on csemCur (left join to keep all rows in new_df)
new_df = new_df.merge(
    donnees_patrimoniales[['csemCur', 'Pression Nominale Relative']], 
    on='csemCur', 
    how='left'
)

# convert csemCur back to Int
new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce').astype('Int64')  # use Int64 to allow for NaN values

# add CR_threshold column based on Pression Nominale Relative
new_df.loc[:, 'CR_threshold'] = new_df['Pression Nominale Relative'] * 0.914

# add age at 2023 column 
# 转换Date MES为datetime（放在end_date定义前）
new_df['Date MES'] = pd.to_datetime(new_df['Date MES'])
end_date = pd.to_datetime('2023-12-31')
new_df['Age 2023'] = (end_date - new_df['Date MES']).dt.days / 365.0

# save to a new CSV
new_df.to_csv('new_df_with_V_MES.csv', index=False)

# results
print("New DataFrame (new_df) created successfully!")
print(f"Shape: {new_df.shape}")
print(f"Unique csemCur count: {new_df['csemCur'].nunique()}")
print(new_df.head())


New DataFrame (new_df) created successfully!
Shape: (595, 17)
Unique csemCur count: 595
      csemCur  year  delta_pressure   cr  \
0  5000200795  2023             NaN  0.0   
1  5000252448  2023             NaN  0.0   
2  5000252449  2023             NaN  0.0   
3  5000252450  2023             NaN  0.0   
4  5000252524  2023             NaN  0.0   

   compensatedRelativePressure_mafiltered  COPER  MANX     alpha    Volume  \
0                                3.628186      0     0  0.004915  0.757573   
1                                3.208579      0     0  0.181613  6.317558   
2                                3.483369      0     0  0.022172  6.302540   
3                                3.452095      0     0  0.018647  9.806732   
4                                3.502114      0     0  0.032852  0.443864   

    Date MES  algo_critere_exploit_B_mafiltered  \
0 1985-01-01                              False   
1 1984-01-01                              False   
2 1984-01-01             

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_89650/777648841.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))
